In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

train = pd.read_csv(BASE_DIR / "data" / "train.csv")
test = pd.read_csv(BASE_DIR / "data" / "test.csv")

print("Размер train:", train.shape)
print("Размер test:", test.shape)
train.head()

В train находится 1460 домов и 81 колонка: `Id`, 79 признаков и целевая переменная `SalePrice`. В test 1459 домов без известной цены.

In [ ]:
train["SalePrice"].describe()

Медианная цена равна 163 000 долларов. Минимальная цена — 34 900, максимальная — 755 000. Большая разница между медианой и максимумом указывает на правый хвост распределения.

## Пропущенные значения

In [ ]:
missing_train = train.isna().sum().sort_values(ascending=False)
missing_test = test.isna().sum().sort_values(ascending=False)

missing = pd.DataFrame({
    "train": missing_train,
    "test": missing_test
}).fillna(0).astype(int)

missing.head(15)

In [ ]:
top_missing = missing_train.head(15).sort_values()

plt.figure(figsize=(9, 6))
top_missing.plot(kind="barh", color="steelblue")
plt.title("Колонки с наибольшим числом пропусков в train")
plt.xlabel("Количество пропусков")
plt.ylabel("Признак")
plt.tight_layout()
plt.show()

Больше всего пропусков находится в `PoolQC`, `MiscFeature`, `Alley` и `Fence`. Для многих категориальных признаков пропуск означает отсутствие объекта, например бассейна или забора. Поэтому категории заполняются отдельным значением `Missing`. Числовые пропуски заполняются медианой, устойчивой к большим значениям.

## Распределение цены

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(train["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("Распределение SalePrice")
axes[0].set_xlabel("Цена, доллары")

sns.histplot(np.log(train["SalePrice"]), kde=True, ax=axes[1])
axes[1].set_title("Распределение log(SalePrice)")
axes[1].set_xlabel("Логарифм цены")

plt.tight_layout()
plt.show()

Исходная цена имеет длинный правый хвост из-за небольшого числа очень дорогих домов. После логарифмирования распределение становится более симметричным. Поэтому все модели обучаются на `log(SalePrice)`, а перед сохранением submission прогноз возвращается в доллары через `np.exp()`.

## Качество, площадь и цена

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=train, x="OverallQual", y="SalePrice", ax=axes[0]
)
axes[0].set_title("Цена в зависимости от общего качества")
axes[0].set_xlabel("OverallQual")
axes[0].set_ylabel("SalePrice")

sns.scatterplot(
    data=train, x="GrLivArea", y="SalePrice",
    alpha=0.7, ax=axes[1]
)
axes[1].set_title("Цена и жилая площадь")
axes[1].set_xlabel("GrLivArea")
axes[1].set_ylabel("SalePrice")

plt.tight_layout()
plt.show()

Чем выше общая оценка качества, тем выше типичная цена дома. С жилой площадью также видна сильная положительная связь, хотя встречаются большие дома с относительно низкой ценой. Эти нелинейные зависимости подходят для моделей на деревьях и бустингов.

## Корреляции числовых признаков

In [ ]:
numeric_data = train.select_dtypes(include="number").drop(columns=["Id"])
correlations = (
    numeric_data.corr()["SalePrice"]
    .drop("SalePrice")
    .sort_values(key=abs, ascending=False)
    .head(12)
)

plt.figure(figsize=(9, 6))
correlations.sort_values().plot(kind="barh", color="darkorange")
plt.title("Числовые признаки с наибольшей корреляцией с ценой")
plt.xlabel("Корреляция с SalePrice")
plt.tight_layout()
plt.show()

correlations

Наиболее сильные числовые связи с ценой имеют `OverallQual`, `GrLivArea`, `GarageCars`, `GarageArea` и `TotalBsmtSF`. При этом в модели оставлены все 79 признаков, потому что слабая отдельная корреляция не означает, что признак бесполезен во взаимодействии с другими.

## Итоговые выводы

- В данных много пропусков, причём в категориальных колонках они часто означают отсутствие объекта.
- Числовые пропуски заполняются медианой, категориальные — значением `Missing`.
- Цена имеет правый хвост, поэтому модель обучается на `log(SalePrice)`.
- Наиболее заметные связи с ценой имеют качество, жилая площадь, гараж и площадь подвала.
- Предобработка включена в pipeline и обучается отдельно внутри каждого фолда.
- Для оценки используется одинаковый 10-fold KFold и Log RMSE.